# End-to-End Data Analytics & Machine Learning Project
## Industry-oriented workflow: Data Gathering → Data Quality → EDA → Analytics → ML → Deployment

**Audience:** Beginner learning professional data workflows  
**Example project:** Real-estate listing analysis and property price prediction  
**Language:** Python | **Environment:** Jupyter Notebook on Windows or any OS

### What this notebook is—and is not
This is a reusable project template with runnable code for a local CSV-based workflow, plus optional API ingestion and FastAPI deployment examples. It does not claim to have scraped a live website, trained a model, or deployed an API before you run it. Real-world source access, schema, permissions, data quality, and model performance must be verified on your actual data.

**Important:** Listing prices are not necessarily actual transaction prices. The model trained here estimates the target present in your dataset, not guaranteed market value.

### Project stages
1. Define the business problem and acceptance criteria.
2. Set up a reproducible environment and project structure.
3. Gather data from approved sources and preserve raw snapshots.
4. Validate schema, data quality, and lineage.
5. Clean and standardize data.
6. Perform exploratory data analysis (EDA).
7. Build business metrics and analytics outputs.
8. Create leakage-safe train/validation/test splits.
9. Train a baseline and candidate models.
10. Evaluate errors and document limitations.
11. Save a versioned model pipeline.
12. Test and expose inference through an API.
13. Plan automation, CI/CD, monitoring, and retraining.

**How to use:** Run cells in order. Read the explanation before running each code cell. If you have no dataset, the notebook can generate clearly labelled synthetic demonstration data; do not present synthetic results as real market insights.

## 0. Project configuration

### Why this matters
Keep paths, target columns, features, and random seeds in one place. Central configuration reduces hidden assumptions and makes runs easier to reproduce.

**Before production:** Replace example column names with your source schema and document every change.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import logging
import platform
import sys
import warnings

PROJECT_ROOT = Path.cwd()  # Run notebook from the project root for predictable paths.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
LOGS_DIR = PROJECT_ROOT / "logs"

for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR, ARTIFACTS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "project_name": "real_estate_analytics_ml",
    "target": "price_pkr",
    "numeric_features": ["area_sqft", "bedrooms"],
    "categorical_features": ["city", "property_type"],
    "test_size": 0.20,
    "validation_size_within_train": 0.20,
    "random_state": 42,
    "synthetic_demo_if_no_data": True,
}
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", PROJECT_ROOT.resolve())
print("Folders created.")

## 1. Business understanding and acceptance criteria

### Define the problem before coding
A useful ML project begins with a business question, intended user, prediction time, and measurable success criteria—not with choosing an algorithm.

**Example objective:** Analyze property listing patterns and estimate a listing price from property attributes.

### Example acceptance criteria (must be agreed with a real stakeholder)
- Data source is authorized and provenance is recorded.
- Required fields have documented meanings and units.
- Data quality issues are quantified; no silent row deletion.
- A simple baseline is evaluated before complex models.
- A final holdout test set is not used for tuning.
- Model error is reported in PKR and by relevant segments.
- API validates inputs and returns a documented response.
- Deployment has health checks, logs, security controls, and rollback instructions.

**Decision:** If a dashboard or a simple rule answers the business question, do not add ML merely to make the project look advanced.

## 2. Environment and dependencies

Install packages in your terminal or in a dedicated virtual environment. Avoid installing/upgrading packages repeatedly inside a production notebook.

Recommended baseline packages:
- pandas, numpy: tabular data
- matplotlib, seaborn: visualization
- scikit-learn: preprocessing, model selection, metrics
- requests: HTTP data collection
- joblib: trusted model artifact persistence
- pytest: automated tests
- fastapi, uvicorn: optional inference API

Optional later: SQLAlchemy + database driver, MLflow for experiment tracking, Great Expectations or Pandera for richer data contracts, Docker for packaging, and an orchestrator such as Prefect/Airflow when workflow complexity justifies it.

In [ ]:
# Uncomment and run in a notebook only if you need to install dependencies.
# %pip install pandas numpy matplotlib seaborn scikit-learn requests joblib
# %pip install fastapi uvicorn pytest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = CONFIG["random_state"]
np.random.seed(SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
print("Core libraries imported.")

## 3. Data source inventory and collection plan

Before collection, record:
- Source name and owner
- URL/database/table/API endpoint
- Collection permission and terms
- Collection date/time and timezone
- Expected schema, units, and row grain
- Pagination/rate limits/authentication
- Refresh frequency and failure policy

For website scraping, respect terms, robots policies, privacy requirements, and rate limits. Do not bypass authentication or technical access controls. For APIs, use official documentation and secrets via environment variables—not hard-coded credentials.

In [ ]:
source_inventory = pd.DataFrame([
    {
        "source_name": "replace_with_approved_source",
        "source_type": "CSV/API/database/website",
        "source_location": "replace_with_url_or_table",
        "permission_checked": False,
        "expected_grain": "one row per listing snapshot",
        "refresh_frequency": "manual / daily / weekly",
        "owner_or_contact": "replace_me",
        "notes": "Document schema, units, pagination, and limitations",
    }
])
source_inventory

## 4. Data ingestion

### Recommended pattern
1. Fetch from an approved source.
2. Apply timeout and HTTP status checks.
3. Check response size/content type and schema.
4. Store a timestamped raw snapshot without silently overwriting prior data.
5. Record checksum, row count, source, and run status.
6. Make reruns safe (idempotent) and handle retries only for transient errors.

The helper below supports a direct CSV URL. Some services return HTML, require authentication, or provide paginated JSON; adapt the connector to the official source contract.

In [ ]:
import requests

def fetch_csv_snapshot(url: str, output_dir: Path = RAW_DIR, timeout=(5, 30)) -> dict:
    """Fetch an approved direct CSV URL and save a timestamped immutable snapshot."""
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()
    content = response.content
    if not content:
        raise ValueError("Downloaded response is empty.")
    if len(content) > 250 * 1024 * 1024:
        raise ValueError("Response exceeds the example 250 MB safety limit.")
    # Content-type varies across providers; inspect and validate the file itself too.
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    digest = hashlib.sha256(content).hexdigest()
    path = output_dir / f"source_snapshot_{timestamp}_{digest[:10]}.csv"
    path.write_bytes(content)

    # Parse once to verify it is readable as CSV.
    preview = pd.read_csv(path, nrows=5)
    return {
        "path": str(path),
        "collected_at_utc": timestamp,
        "sha256": digest,
        "bytes": len(content),
        "content_type": content_type,
        "preview_columns": preview.columns.tolist(),
    }

# Example usage (only after you have an approved direct CSV URL):
# ingestion_result = fetch_csv_snapshot("https://example.com/data.csv")
# ingestion_result

### Local CSV/Excel input (the usual beginner path)

Put your actual source file in `data/raw/`. The loader supports CSV and Excel. Keep original source files unchanged; use processed copies for cleaning. If the file is very large, use chunked ingestion or a database rather than loading everything into RAM.

In [ ]:
def load_local_file(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file extension: {suffix}")

# Example:
# df_raw = load_local_file(RAW_DIR / "your_file.csv")

## 5. No-dataset fallback: synthetic demonstration data

This optional section creates fake records so the notebook can be run end-to-end without internet access. **Synthetic data is only for testing code**; it cannot support real business conclusions, market claims, or meaningful production model quality claims.

If you have a real dataset, load it instead and skip the synthetic generator.

In [ ]:
def make_synthetic_demo(n: int = 1500, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    cities = rng.choice(["Karachi", "Lahore", "Islamabad", "Rawalpindi"], n,
                        p=[0.42, 0.28, 0.18, 0.12])
    types = rng.choice(["house", "flat", "plot"], n, p=[0.45, 0.35, 0.20])
    area = np.clip(rng.lognormal(mean=7.1, sigma=0.55, size=n), 250, 12000)
    bedrooms = np.where(types == "plot", 0, rng.integers(1, 7, size=n))
    city_effect = pd.Series(cities).map({
        "Karachi": 1.00, "Lahore": 0.90, "Islamabad": 1.35, "Rawalpindi": 0.72
    }).to_numpy()
    type_effect = pd.Series(types).map({
        "house": 1.25, "flat": 0.95, "plot": 0.60
    }).to_numpy()
    noise = rng.normal(0, 1_500_000, n)
    price = np.maximum(500_000, area * 14_000 * city_effect * type_effect
                       + bedrooms * 1_200_000 + noise)
    data = pd.DataFrame({
        "listing_id": [f"demo_{i:05d}" for i in range(n)],
        "city": cities,
        "property_type": types,
        "area_sqft": area.round(0),
        "bedrooms": bedrooms,
        "price_pkr": price.round(0),
        "source": "synthetic_demo",
        "collected_at": datetime.now(timezone.utc).isoformat(),
    })
    # Deliberately add a few missing values to demonstrate quality checks.
    data.loc[rng.choice(data.index, size=max(1, n // 100), replace=False), "area_sqft"] = np.nan
    return data

USE_SYNTHETIC_DEMO = CONFIG["synthetic_demo_if_no_data"]
DATA_FILE = RAW_DIR / "properties.csv"

if DATA_FILE.exists():
    df_raw = load_local_file(DATA_FILE)
    DATA_ORIGIN = "user_provided_file"
elif USE_SYNTHETIC_DEMO:
    df_raw = make_synthetic_demo()
    DATA_ORIGIN = "synthetic_demo_only"
else:
    raise FileNotFoundError(
        f"Place a real dataset at {DATA_FILE} or explicitly enable synthetic demo data."
    )

print("Origin:", DATA_ORIGIN)
print("Shape:", df_raw.shape)
display(df_raw.head())

## 6. Raw data profiling and data contract

A **data contract** defines expected columns, types, constraints, and semantics. Validate the incoming data before cleaning. Do not silently assume that a source has the same schema forever.

In [ ]:
EXPECTED_COLUMNS = {
    "city", "property_type", "area_sqft", "bedrooms", "price_pkr"
}

missing_columns = EXPECTED_COLUMNS - set(df_raw.columns)
if missing_columns:
    raise ValueError(f"Required columns missing: {sorted(missing_columns)}")

print("Rows, columns:", df_raw.shape)
display(df_raw.dtypes.rename("dtype").to_frame())
display(df_raw.describe(include="all").T)

In [ ]:
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "distinct_non_null": df.nunique(dropna=True),
    }).sort_values("missing_pct", ascending=False)

quality_profile = profile_dataframe(df_raw)
display(quality_profile)
quality_profile.to_csv(REPORTS_DIR / "raw_quality_profile.csv")

### Duplicate and range checks

Distinguish exact duplicate rows from duplicate business entities. The same property may legitimately appear in multiple listing snapshots or sources. Do not deduplicate on city + area alone: that can remove distinct properties.

In [ ]:
exact_duplicate_count = int(df_raw.duplicated().sum())
print("Exact duplicate rows:", exact_duplicate_count)

range_checks = {
    "non_positive_price": int((pd.to_numeric(df_raw["price_pkr"], errors="coerce") <= 0).sum()),
    "non_positive_area": int((pd.to_numeric(df_raw["area_sqft"], errors="coerce") <= 0).sum()),
    "negative_bedrooms": int((pd.to_numeric(df_raw["bedrooms"], errors="coerce") < 0).sum()),
}
print(json.dumps(range_checks, indent=2))

## 7. Cleaning and standardization

### Principles
- Preserve `df_raw`; transform a copy.
- Parse values according to source-specific rules.
- Standardize category labels using a reviewed mapping.
- Flag invalid values and report their counts.
- Do not automatically remove statistical outliers without investigating them.
- Avoid target leakage: do not create input features using information unavailable at prediction time.

In [ ]:
def clean_properties(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = raw.copy()
    df.columns = (
        df.columns.astype(str).str.strip().str.lower()
          .str.replace(r"[^a-z0-9]+", "_", regex=True)
          .str.strip("_")
    )

    required = ["city", "property_type", "area_sqft", "bedrooms", "price_pkr"]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns after normalization: {sorted(missing)}")

    for col in ["city", "property_type"]:
        df[col] = df[col].astype("string").str.strip().str.lower()
        df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "none": pd.NA})

    for col in ["area_sqft", "bedrooms", "price_pkr"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Create explicit quality flags before deciding what to exclude.
    df["invalid_price_flag"] = df["price_pkr"].le(0).fillna(False)
    df["invalid_area_flag"] = df["area_sqft"].le(0).fillna(False)
    df["missing_city_flag"] = df["city"].isna()
    df["missing_type_flag"] = df["property_type"].isna()

    # Remove exact duplicate rows only. Entity-level deduplication requires a source-specific key.
    df = df.drop_duplicates().copy()

    # Quarantine clearly invalid targets; preserve a report rather than silently deleting.
    quarantine_mask = (
        df["price_pkr"].isna()
        | df["invalid_price_flag"]
        | df["missing_city_flag"]
        | df["missing_type_flag"]
    )
    quarantine = df.loc[quarantine_mask].copy()
    usable = df.loc[~quarantine_mask].copy()

    # Area must be positive to be useful for this example; leave missing area for model imputation.
    usable.loc[usable["invalid_area_flag"], "area_sqft"] = np.nan
    usable.loc[usable["bedrooms"] < 0, "bedrooms"] = np.nan

    return usable, quarantine

df_clean, df_quarantine = clean_properties(df_raw)
print("Raw rows:", len(df_raw))
print("Usable rows:", len(df_clean))
print("Quarantined rows:", len(df_quarantine))
df_quarantine.to_csv(INTERIM_DIR / "quarantined_records.csv", index=False)
df_clean.to_parquet(INTERIM_DIR / "clean_properties.parquet", index=False)
display(df_clean.head())

## 8. Feature definitions and analytics dataset

Feature definitions should be documented, stable, and available at prediction time. For example, price per square foot is useful for descriptive analysis, but because it is calculated using price, it must **not** be used to predict that same price.

In [ ]:
# Descriptive metric only; never use this target-derived value as a predictor for price_pkr.
df_analytics = df_clean.copy()
valid_area = df_analytics["area_sqft"].gt(0)
df_analytics["price_per_sqft"] = np.where(
    valid_area, df_analytics["price_pkr"] / df_analytics["area_sqft"], np.nan
)

analytics_summary = (
    df_analytics.groupby(["city", "property_type"], dropna=False)
    .agg(
        listing_count=("listing_id", "nunique") if "listing_id" in df_analytics.columns
        else ("price_pkr", "size"),
        median_price_pkr=("price_pkr", "median"),
        mean_price_pkr=("price_pkr", "mean"),
        median_area_sqft=("area_sqft", "median"),
        median_price_per_sqft=("price_per_sqft", "median"),
    )
    .reset_index()
    .sort_values("median_price_pkr", ascending=False)
)
display(analytics_summary)
analytics_summary.to_csv(PROCESSED_DIR / "city_property_summary.csv", index=False)

## 9. Exploratory Data Analysis (EDA)

EDA should answer business questions, not merely generate charts. Investigate distributions, missingness, segment differences, potential confounders, outliers, and time patterns when timestamps are reliable.

**Caution:** If the active data origin is synthetic, the charts demonstrate code behavior only—not actual real-estate market findings.

In [ ]:
print("Data origin:", DATA_ORIGIN)
print("Rows and columns:", df_analytics.shape)
display(df_analytics[["price_pkr", "area_sqft", "bedrooms"]].describe().T)

fig, ax = plt.subplots()
df_analytics["price_pkr"].dropna().plot(kind="hist", bins=40, ax=ax)
ax.set_title("Listing price distribution")
ax.set_xlabel("Price (PKR)")
ax.set_ylabel("Listings")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
df_analytics.boxplot(column="price_pkr", by="city", ax=ax, rot=30)
ax.set_title("Price distribution by city")
ax.set_xlabel("City")
ax.set_ylabel("Price (PKR)")
plt.suptitle("")
plt.tight_layout()
plt.show()

display(
    df_analytics.groupby("city")["price_pkr"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)

### EDA interpretation template

For each chart, write:
1. **Observation:** What pattern is visible?
2. **Validation:** Could missingness, duplicates, unit differences, or sampling bias explain it?
3. **Business meaning:** Why might this matter?
4. **Action:** What additional data or analysis is needed?

Correlation is not causation. Public listing data may be biased toward properties that are advertised online, and prices may not represent completed sales.

In [ ]:
eda_notes = {
    "data_origin": DATA_ORIGIN,
    "key_observations": [
        "Write observations after inspecting the actual chart.",
        "Separate measured facts from hypotheses.",
        "Document sampling and source limitations."
    ],
    "limitations": [
        "Listing price may differ from transaction price.",
        "Missing attributes may not be random.",
        "Coverage may vary by source, city, and property type."
    ]
}
(REPORTS_DIR / "eda_notes.json").write_text(
    json.dumps(eda_notes, indent=2), encoding="utf-8"
)

## 10. Leakage-safe train, validation, and test split

### Why split strategy matters
- **Train:** Fit preprocessing and model parameters.
- **Validation / cross-validation:** Select models and tune hyperparameters.
- **Test:** One final, unbiased evaluation after model selection.

If multiple rows refer to the same property, group-aware splitting may be required. If the use case is future prediction, prefer a time-based split. The random split below is an illustrative default for independent rows only; change it when your data structure requires it.

In [ ]:
TARGET = CONFIG["target"]
NUMERIC_FEATURES = CONFIG["numeric_features"]
CATEGORICAL_FEATURES = CONFIG["categorical_features"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing = set([TARGET] + FEATURES) - set(df_clean.columns)
if missing:
    raise ValueError(f"Configure feature names to match your dataset: {sorted(missing)}")

model_data = df_clean.dropna(subset=[TARGET]).copy()
X = model_data[FEATURES].copy()
y = model_data[TARGET].astype(float).copy()

# Example only: use group/time split when observations are dependent or temporal.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=CONFIG["validation_size_within_train"],
    random_state=SEED
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

## 11. Preprocessing and candidate models

Use a scikit-learn `Pipeline` so imputation, scaling, encoding, and estimation are fitted together. This prevents preprocessing from being fit on the held-out test set and makes inference consistent.

The features below are examples. Review whether each feature is available at prediction time and whether the categories and measurements are reliable.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

candidate_models = {
    "baseline_median": DummyRegressor(strategy="median"),
    "ridge": Ridge(alpha=1.0),
    "random_forest": RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=3,
        random_state=SEED,
        n_jobs=-1,
    ),
}

pipelines = {
    name: Pipeline([
        ("preprocessing", preprocessor),
        ("model", estimator),
    ])
    for name, estimator in candidate_models.items()
}

## 12. Train and compare on validation data

Fit candidates on training data and compare on validation data. Keep the test set untouched. MAE is in the target's unit (PKR); RMSE penalizes large errors more heavily. R² is useful context but can be negative and should not be treated as the only success criterion.

In [ ]:
def regression_metrics(y_true, y_pred) -> dict:
    return {
        "MAE_PKR": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_PKR": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

validation_rows = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    validation_rows.append({"model": name, **regression_metrics(y_val, pred)})

validation_results = pd.DataFrame(validation_rows).sort_values("MAE_PKR")
display(validation_results)

best_name = validation_results.iloc[0]["model"]
best_pipeline = pipelines[best_name]
print("Selected candidate based on validation MAE:", best_name)

### Model selection policy
The lowest validation MAE is a simple selection rule for this demonstration, not a universal production policy. Consider segment errors, operational latency, interpretability, stability, data leakage, and business cost. For robust comparisons, use cross-validation on the training portion and tune hyperparameters without touching the test set.

## 13. Final fit and one-time test evaluation

Once the model and configuration are selected, refit the chosen pipeline on train + validation data, then evaluate once on the test set. Do not continue tuning based on test results; if you do, the test set is no longer a clean final holdout.

In [ ]:
final_pipeline = pipelines[best_name]
final_pipeline.fit(X_trainval, y_trainval)
test_predictions = final_pipeline.predict(X_test)
test_metrics = regression_metrics(y_test, test_predictions)

print("Final model:", best_name)
print(json.dumps(test_metrics, indent=2))
pd.DataFrame([{"model": best_name, **test_metrics}]).to_csv(
    REPORTS_DIR / "final_test_metrics.csv", index=False
)

In [ ]:
residuals = y_test.to_numpy() - test_predictions
fig, ax = plt.subplots()
ax.scatter(test_predictions, residuals, alpha=0.5)
ax.axhline(0, linestyle="--")
ax.set_title("Test residuals vs predicted price")
ax.set_xlabel("Predicted price (PKR)")
ax.set_ylabel("Actual - predicted (PKR)")
plt.tight_layout()
plt.show()

error_report = X_test.copy()
error_report["actual_price_pkr"] = y_test
error_report["predicted_price_pkr"] = test_predictions
error_report["absolute_error_pkr"] = np.abs(residuals)
display(error_report.sort_values("absolute_error_pkr", ascending=False).head(10))

## 14. Segment-level evaluation and error analysis

An acceptable overall MAE can hide poor performance for a city or property type with fewer examples. Analyze segment performance where sample size is adequate; clearly flag small groups.

In [ ]:
segment_eval = error_report.copy()
segment_eval["city"] = X_test["city"].to_numpy()
segment_eval["property_type"] = X_test["property_type"].to_numpy()

segment_metrics = (
    segment_eval.groupby(["city", "property_type"])
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "MAE_PKR": mean_absolute_error(g["actual_price_pkr"], g["predicted_price_pkr"]),
            "RMSE_PKR": np.sqrt(mean_squared_error(g["actual_price_pkr"], g["predicted_price_pkr"])),
        }),
        include_groups=False
    )
    .reset_index()
)
display(segment_metrics.sort_values("MAE_PKR", ascending=False))
segment_metrics.to_csv(REPORTS_DIR / "segment_metrics.csv", index=False)

## 15. Persist model artifact and metadata

A model artifact should be accompanied by metadata: target, feature list, selected model, evaluation metrics, training timestamp, data provenance, code version, and dependency versions. Keep personally identifiable or sensitive raw data out of metadata and logs.

In [ ]:
import joblib
import sklearn

model_path = ARTIFACTS_DIR / "property_price_pipeline.joblib"
metadata_path = ARTIFACTS_DIR / "model_metadata.json"

joblib.dump(final_pipeline, model_path)

metadata = {
    "model_name": best_name,
    "target": TARGET,
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "test_metrics": test_metrics,
    "data_origin": DATA_ORIGIN,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__,
    "random_seed": SEED,
    "artifact_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
    "limitations": [
        "Only estimates the dataset target, which may be listing price rather than transaction price.",
        "Performance depends on source coverage, data quality, and feature availability.",
        "Synthetic demonstration results are not evidence of real-world performance."
    ]
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved model:", model_path)
print("Saved metadata:", metadata_path)

## 16. Inference smoke test

A smoke test checks that the saved pipeline can load and produce a prediction from one valid record. Do not load untrusted pickle/joblib artifacts: deserialization can execute code. Use trusted, access-controlled artifacts and compatible dependency versions.

In [ ]:
loaded_pipeline = joblib.load(model_path)
sample_input = X_test.iloc[[0]].copy()
sample_prediction = float(loaded_pipeline.predict(sample_input)[0])

print("Sample input:")
display(sample_input)
print(f"Predicted target: {sample_prediction:,.2f} PKR")
assert np.isfinite(sample_prediction), "Prediction is not finite."

## 17. Optional FastAPI inference service

For deployment, place API code in `api/main.py` and run it as a service; do not keep a notebook kernel as the production server. The following is a template. Adapt validation to the actual schema and do not expose it publicly without appropriate authentication, TLS, rate limits, logging, and operational controls.

Create `api/main.py` using the code below, or copy it from this notebook. It expects the model artifact at `artifacts/property_price_pipeline.joblib` relative to the project root.

In [ ]:
# Save the following service source to api/main.py when you are ready to deploy.
api_source = r'''
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

ROOT = Path(__file__).resolve().parents[1]
MODEL_PATH = ROOT / "artifacts" / "property_price_pipeline.joblib"

app = FastAPI(title="Property Price Prediction API", version="1.0.0")
model = joblib.load(MODEL_PATH)

class PropertyInput(BaseModel):
    area_sqft: float = Field(gt=0, le=1_000_000)
    bedrooms: int = Field(ge=0, le=100)
    city: str = Field(min_length=1, max_length=100)
    property_type: str = Field(min_length=1, max_length=50)

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.post("/predict")
def predict(payload: PropertyInput):
    row = pd.DataFrame([payload.model_dump()])
    try:
        value = float(model.predict(row)[0])
    except Exception as exc:
        # In production, log the exception safely with a request ID.
        raise HTTPException(status_code=422, detail="Input could not be processed.") from exc
    return {"predicted_price_pkr": value, "model_version": "1.0.0"}
'''
api_dir = PROJECT_ROOT / "api"
api_dir.mkdir(exist_ok=True)
(api_dir / "main.py").write_text(api_source, encoding="utf-8")
print("Wrote API template to:", api_dir / "main.py")

### Run the API locally

Install FastAPI and Uvicorn in the project environment, then run from the project root:

```bash
python -m pip install fastapi uvicorn
uvicorn api.main:app --host 127.0.0.1 --port 8000
```

Open `http://127.0.0.1:8000/docs` locally to test requests. Binding to `127.0.0.1` keeps the service local. For production, configure an appropriate server/container, HTTPS, access controls, secrets, structured logging, health/readiness checks, and safe deployment/rollback.

**Important schema check:** The training feature names must match the API payload columns. If you change features, update the API schema and tests together.

## 18. Automated tests

At minimum, test:
- Required source columns and data types
- Cleaning behavior for missing/malformed values
- Feature contract matches training and inference
- Model artifact loads from a trusted location
- Predictions are finite and response schema is stable
- API returns expected errors for invalid requests

Example `pytest` test (save as `tests/test_model.py`):

In [ ]:
test_source = r'''
import numpy as np
import joblib
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]

def test_model_artifact_predicts_finite_value():
    model = joblib.load(ROOT / "artifacts" / "property_price_pipeline.joblib")
    sample = {
        "area_sqft": 1200.0,
        "bedrooms": 3,
        "city": "karachi",
        "property_type": "house",
    }
    import pandas as pd
    prediction = model.predict(pd.DataFrame([sample]))
    assert len(prediction) == 1
    assert np.isfinite(prediction[0])
'''
tests_dir = PROJECT_ROOT / "tests"
tests_dir.mkdir(exist_ok=True)
(tests_dir / "test_model.py").write_text(test_source, encoding="utf-8")
print("Wrote example test to:", tests_dir / "test_model.py")

Run from the project root after the model artifact exists:

```bash
python -m pip install pytest
pytest -q
```

A production test suite should include data-contract, API integration, security, dependency, and regression tests; the single smoke test above is only a starting point.

## 19. Orchestration and scheduled runs

For a small project, a Python entry-point script or Windows Task Scheduler may be sufficient. Use Airflow or Prefect only when dependencies, retries, schedules, observability, and team workflows justify the added operational complexity.

Recommended task boundaries:
1. `ingest`: collect and snapshot source data
2. `validate`: enforce schema and quality checks
3. `transform`: create cleaned/curated datasets
4. `analytics`: refresh business metrics
5. `train`: train candidate models when required
6. `evaluate`: compare against acceptance criteria
7. `release`: publish only approved artifacts
8. `monitor`: log pipeline health and model quality

Each task should have clear inputs/outputs, structured logs, failure behavior, and safe rerun semantics. Do not overwrite a working production model just because a scheduled training job finished.

## 20. Monitoring and retraining policy

### Operational monitoring
Track job success/failure, data freshness, row counts, schema changes, API latency/error rates, CPU/memory, and artifact versions.

### Model monitoring
When actual outcomes become available, track MAE/RMSE over time and by important segment. Input drift is a signal to investigate, not automatic proof that model performance has degraded.

### Retraining policy
- Trigger a review when source schema or input distributions materially change.
- Retrain using validated new data and a documented split strategy.
- Compare the candidate with the current model.
- Promote only when pre-agreed gates pass.
- Keep the previous artifact and rollback procedure.
- Record who approved the release, model/data/code versions, and evaluation results.

## 21. Security, privacy, and governance checklist

- Do not commit `.env`, API tokens, passwords, customer identifiers, or private data.
- Use least-privilege credentials and environment-based secrets.
- Respect source permissions, licenses, terms, privacy rules, and retention policies.
- Minimize collection and logging of personal information.
- Validate inputs and limit request sizes.
- Use TLS and authentication for remotely accessible APIs.
- Keep dependencies updated and scan for known vulnerabilities.
- Treat model artifacts as executable/trusted files; control who can write and load them.
- Document data provenance, model limitations, intended use, and prohibited use.
- Back up critical data and define retention/deletion policies.

Example `.gitignore` entries:

In [ ]:
gitignore_text = '''
.venv/
__pycache__/
.ipynb_checkpoints/
.env
*.log
data/raw/*
data/interim/*
data/processed/*
artifacts/*
!data/raw/.gitkeep
!data/interim/.gitkeep
!data/processed/.gitkeep
!artifacts/.gitkeep
'''
(PROJECT_ROOT / ".gitignore").write_text(gitignore_text.strip() + "\n", encoding="utf-8")
print("Wrote .gitignore template.")

## 22. Final deliverables and project handover

A professional handover should include:
- `README.md`: setup, run instructions, data source, outputs, limitations
- `docs/requirements.md`: business problem and acceptance criteria
- `docs/data_dictionary.md`: fields, units, definitions, lineage
- `reports/`: quality profile, EDA notes, model metrics, segment evaluation
- `src/`: reusable ingestion, cleaning, analytics, training and inference code
- `tests/`: automated checks
- `artifacts/`: trusted, versioned model pipeline and metadata
- `api/`: API implementation and schema
- Deployment instructions, monitoring plan, rollback steps, and ownership

### Suggested README outline
1. Project summary and intended users
2. Problem definition and success metrics
3. Data sources, permission, provenance, and limitations
4. Setup and dependency installation
5. How to run each pipeline stage
6. Data quality and EDA findings
7. Model choice and final test metrics
8. API usage and sample request
9. Testing, deployment, monitoring, rollback
10. Known limitations and future improvements

## 23. Production readiness review

Before calling the project “done”, verify:
- [ ] The source and its usage are approved.
- [ ] Data lineage and schemas are documented.
- [ ] Pipeline reruns are safe and failures are observable.
- [ ] Cleaning rules are tested and raw data is preserved.
- [ ] Analytics definitions have been checked.
- [ ] Leakage-safe validation and untouched final testing were used.
- [ ] Baseline and candidate metrics are documented.
- [ ] Segment-level errors and limitations are understood.
- [ ] Model, data, code, and dependencies are traceable.
- [ ] API validation, security, health checks, and logging are implemented.
- [ ] Deployment and rollback are tested.
- [ ] Monitoring and retraining criteria have owners.

## 24. What to do next

**Recommended implementation order:**
1. Run this notebook end-to-end using the synthetic demo to learn the code path.
2. Replace the demo with an authorized real dataset and update the schema/configuration.
3. Move stable notebook code into small reusable Python modules.
4. Add tests and a repeatable run command.
5. Build the analytics dashboard.
6. Validate ML quality with a suitable split strategy and business acceptance criteria.
7. Serve the approved model through FastAPI.
8. Containerize and deploy only when local tests and security checks pass.

**Do not skip data understanding or evaluation just to reach deployment quickly.** A professional pipeline is one whose results are traceable, repeatable, tested, and honest about limitations—not simply one with many tools.